### **Notebook 9: Procesamiento de variables derivadas**

##### **Objetivo:** Generar variables que enriquecen la investigación, aprovechando el valor predictivo de las variables originales.


##### **1. Justificación Clínica y Operativa**
Para maximizar la capacidad predictiva de los algoritmos de Machine Learning sobre el costo y la severidad de los episodios GRD, se construyeron variables derivadas que capturan la "intensidad" del caso clínico mejor que los datos tabulares crudos.

A. Variables de Carga de Enfermedad (Comorbilidades)
* **`NUM_COMORBILIDADES`:** Conteo de diagnósticos secundarios válidos (excluyendo el cáncer base y códigos administrativos). Un paciente con múltiples comorbilidades (ej. diabetes + EPOC + falla renal) tiene un riesgo inherentemente mayor de complicaciones postoperatorias y estadías prolongadas.
* **`CARGA_ONCOLOGICA`:** Conteo estricto de códigos neoplásicos (C00-C97, D00-D48). Diferencia a un paciente con un tumor primario único de uno con enfermedad metastásica multisistémica.
* **`COMORBILIDAD_PRINCIPAL`:** Mapeo de la primera comorbilidad válida a los 22 Macrogrupos del CIE-10 (ej. *Endocrinas y Metabólicas*). Para el grupo control (no oncológico), se asigna la etiqueta "NO_APLICA", ya que su diagnóstico principal ya define su condición base.

B. Variables de Intensidad Terapéutica
* **`NUM_PROCEDIMIENTOS`:** Sumatoria de intervenciones realizadas (máximo 30). Altamente correlacionado con el uso de insumos, horas de pabellón y complejidad de enfermería.
* **`CANTIDAD_TRASLADOS`:** Métrica de inestabilidad clínica. Cuantifica los movimientos del paciente entre distintas unidades del hospital (ej. Urgencia -> Pabellón -> UCI -> Sala Básica).

C. Variables Temporales y Demográficas
* **`EDAD`:** Calculada en años cronológicos al momento del ingreso. Es el predictor demográfico primario de riesgo de mortalidad.
* **`DIAS_ESTADIA`:** Diferencia en días entre la fecha de alta y la fecha de ingreso. Representa el consumo principal del recurso "día-cama".

##### **2. Análisis de Calidad y Manejo de Nulos (Anomalías Temporales)**
Al calcular las variables numéricas, las columnas de conteo (`NUM_COMORBILIDADES`, `CANTIDAD_TRASLADOS`, etc.) presentaron un **0% de valores nulos**. 

Sin embargo, el cruce de fechas para calcular `EDAD` y `DIAS_ESTADIA` reveló inconsistencias de digitación administrativa en el origen de los datos MINSAL (ej. fechas de alta anteriores a la fecha de ingreso, o ingresos maternos heredados a recién nacidos generando edades negativas).
* **Filtro de Corrección:** Se aplicó un algoritmo de parseo estricto adaptado a la mutación de formatos del MINSAL (paso de `yyyy-mm-dd` a `dd-mm-yyyy` en 2023), rescatando más de 1 millón de registros anuales.
* **Tratamiento de Errores Residuales:** Las anomalías negativas reales se transformaron a nulos (`NaN`). La auditoría final arrojó un volumen estadísticamente insignificante de errores irrecuperables (Máximo histórico: 53 casos nulos en `EDAD` y 50 en `DIAS_ESTADIA` durante 2024, sobre un total de >1.076.000 egresos).
* **Veredicto:** **Eliminación Directa**. Dado que estos valores nulos representan consistentemente menos del **0.005%** de la cohorte anual, se procede a la eliminación de las filas (Listwise Deletion). La imputación estadística no se justifica y podría introducir sesgos al crear tiempos de estadía sintéticos en episodios con fechas corruptas.

In [ ]:
import pandas as pd # Librería para manipulación de datos
import numpy as np # Librería para operaciones numéricas
import os # Librería para manejo de rutas y archivos
import re # Librería para expresiones regulares


# ============================================================================
# 1. DEFINICIÓN DE RUTAS
# ============================================================================

# Define la carpeta donde se encuentran los archivos procesados.
carpeta_procesados = "../../Datos/Datos procesados"

# Genera automáticamente los nombres de los archivos correspondientes a 2019-2024.
archivos_procesados = [
    f"GRD_PROCESADO_{año}.csv"
    for año in range(2019, 2025)
]

# Informa el inicio del proceso de creación de variables derivadas.
print(
    "Iniciando creación de variables derivadas...\n"
    + "=" * 60
)


# ============================================================================
# 2. CONFIGURACIÓN PARA COMORBILIDADES
# ============================================================================

# Define letras CIE-10 que corresponden a códigos administrativos y no a comorbilidades.
letras_excluidas = ['Z', 'V', 'W', 'X', 'Y']


# Define los grupos de letras CIE-10 utilizados para clasificar las comorbilidades.
condiciones_macro_letras = [
    ['A', 'B'],
    ['D'],
    ['E'],
    ['F'],
    ['G'],
    ['H'],
    ['I'],
    ['J'],
    ['K'],
    ['L'],
    ['M'],
    ['N'],
    ['O', 'P', 'Q'],
    ['R'],
    ['S', 'T'],
    ['U']
]


# Define el nombre de cada macro-categoría de comorbilidad.
opciones_macro = [
    "INFECCIOSAS Y PARASITARIAS",
    "SANGRE E INMUNIDAD",
    "ENDOCRINAS Y METABOLICAS",
    "TRASTORNOS MENTALES",
    "SISTEMA NERVIOSO",
    "OJO Y OIDO",
    "SISTEMA CIRCULATORIO",
    "SISTEMA RESPIRATORIO",
    "SISTEMA DIGESTIVO",
    "PIEL Y TEJIDO SUBCUTANEO",
    "SISTEMA MUSCULOESQUELETICO",
    "SISTEMA GENITOURINARIO",
    "MATERNO-INFANTILES Y CONGENITAS",
    "SINTOMAS Y HALLAZGOS",
    "TRAUMATISMOS Y ENVENENAMIENTOS",
    "CODIGOS PROVISIONALES (COVID)"
]


# ============================================================================
# FUNCIÓN 1: OBTENER MACRO-CATEGORÍA DE COMORBILIDAD
# ============================================================================

def obtener_macro_comorbilidad(letra):
    """
    Definición: Identifica la macro-categoría CIE-10 asociada a una letra.

    Entrada:
        letra: Letra inicial del código CIE-10.

    Salida:
        Retorna el nombre de la macro-categoría correspondiente.
        Si la letra no pertenece a ninguna categoría, retorna np.nan.
    """

    # Recorre simultáneamente los grupos de letras y sus categorías.
    for letras_cond, macro in zip(
        condiciones_macro_letras,
        opciones_macro
    ):

        # Comprueba si la letra pertenece al grupo actual.
        if letra in letras_cond:

            # Devuelve la macro-categoría correspondiente.
            return macro

    # Devuelve un valor nulo si no se encontró ninguna categoría.
    return np.nan


# ============================================================================
# 3. PROCESAMIENTO DE LOS ARCHIVOS
# ============================================================================

# Recorre todos los archivos procesados.
for archivo in archivos_procesados:

    # Construye la ruta completa del archivo de origen.
    ruta_origen = os.path.join(
        carpeta_procesados,
        archivo
    )

    # Extrae el año desde el nombre del archivo.
    año = archivo[-8:-4]

    # Carga el dataset completo en memoria.
    df = pd.read_csv(
        ruta_origen,
        low_memory=False
    )


    # =========================================================================
    # A. CANTIDAD DE TRASLADOS INTERNOS
    # =========================================================================

    # Genera los nombres de las nueve columnas de traslados internos.
    cols_traslados = [
        f"SERVICIOTRASLADO{i}"
        for i in range(1, 10)
    ]

    # Cuenta cuántos traslados registrados tiene cada paciente.
    df['CANTIDAD_TRASLADOS'] = (
        df[cols_traslados]
        .notnull()
        .sum(axis=1)
    )


    # =========================================================================
    # B. NÚMERO DE PROCEDIMIENTOS
    # =========================================================================

    # Genera los nombres de las treinta columnas de procedimientos.
    cols_procedimientos = [
        f"PROCEDIMIENTO{i}"
        for i in range(1, 31)
    ]

    # Cuenta cuántos procedimientos registrados tiene cada paciente.
    df['NUM_PROCEDIMIENTOS'] = (
        df[cols_procedimientos]
        .notnull()
        .sum(axis=1)
    )


    # =========================================================================
    # C. CARGA ONCOLÓGICA, COMORBILIDADES Y COMORBILIDAD PRINCIPAL
    # =========================================================================

    # Genera los nombres de las 35 columnas de diagnósticos.
    cols_diagnosticos = [
        f"DIAGNOSTICO{i}"
        for i in range(1, 36)
    ]


    # =========================================================================
    # FUNCIÓN 2: PROCESAR DIAGNÓSTICOS DE CADA PACIENTE
    # =========================================================================

    def procesar_diagnosticos(fila):
        """
        Definición: Analiza los diagnósticos CIE-10 de un paciente y calcula
        su carga oncológica, número de comorbilidades y comorbilidad principal.

        Entrada:
            fila: Serie de pandas correspondiente a un registro del dataset.

        Salida:
            Retorna una Serie con tres valores:
                1. CARGA_ONCOLOGICA
                2. NUM_COMORBILIDADES
                3. COMORBILIDAD_PRINCIPAL
        """

        # Inicializa el contador de diagnósticos relacionados con cáncer.
        cancer = 0

        # Inicializa el contador de comorbilidades.
        comorbilidad = 0

        # Inicializa la categoría de la comorbilidad principal.
        comorbilidad_principal = np.nan

        # Determina si el paciente pertenece a la cohorte oncológica.
        es_oncologico = (
            fila.get(
                'CATEGORIA_CANCER',
                'SIN_CANCER'
            ) != 'SIN_CANCER'
        )


        # ---------------------------------------------------------------------
        # RECORRIDO DE LOS 35 DIAGNÓSTICOS
        # ---------------------------------------------------------------------

        # Recorre todas las columnas de diagnóstico disponibles.
        for col in cols_diagnosticos:

            # Obtiene el valor del diagnóstico actual.
            val = fila[col]

            # Ignora diagnósticos que no tienen información.
            if pd.isna(val):
                continue

            # Convierte el diagnóstico a texto, elimina espacios y usa mayúsculas.
            val_str = (
                str(val)
                .strip()
                .upper()
            )

            # Ignora valores que representan diagnósticos desconocidos o faltantes.
            if val_str in [
                "DESCONOCIDO",
                "S/D",
                ""
            ]:
                continue

            # Extrae la letra y los dos primeros dígitos del código CIE-10.
            extraccion = re.match(
                r'^([A-Z])(\d{2})',
                val_str
            )

            # Ignora valores que no tienen una estructura CIE-10 reconocible.
            if not extraccion:
                continue

            # Extrae la letra inicial del código CIE-10.
            letra = extraccion.group(1)

            # Extrae los dos primeros dígitos del código CIE-10.
            numero = float(
                extraccion.group(2)
            )


            # -----------------------------------------------------------------
            # CLASIFICACIÓN CIE-10
            # -----------------------------------------------------------------

            # Identifica códigos relacionados con cáncer.
            es_codigo_cancer = (
                letra == 'C'
                or (
                    letra == 'D'
                    and numero <= 48
                )
            )

            # Identifica códigos administrativos que deben excluirse.
            es_codigo_admin = (
                letra in letras_excluidas
            )


            # -----------------------------------------------------------------
            # CLASIFICACIÓN DEL DIAGNÓSTICO
            # -----------------------------------------------------------------

            # Si corresponde a cáncer, incrementa la carga oncológica.
            if es_codigo_cancer:

                # Incrementa el contador de diagnósticos oncológicos.
                cancer += 1

            # Si no es cáncer ni un código administrativo, se considera comorbilidad.
            elif not es_codigo_admin:

                # Incrementa el contador de comorbilidades.
                comorbilidad += 1

                # Captura la primera comorbilidad válida del paciente oncológico.
                if (
                    es_oncologico
                    and pd.isna(comorbilidad_principal)
                ):

                    # Obtiene la macro-categoría asociada a la letra CIE-10.
                    comorbilidad_principal = (
                        obtener_macro_comorbilidad(letra)
                    )


        # ---------------------------------------------------------------------
        # ASIGNACIONES FINALES
        # ---------------------------------------------------------------------

        # Los pacientes no oncológicos no tienen comorbilidad principal aplicable.
        if not es_oncologico:

            # Asigna la categoría correspondiente a pacientes no oncológicos.
            comorbilidad_principal = "NO_APLICA"

        # Identifica pacientes oncológicos sin una comorbilidad principal válida.
        elif pd.isna(comorbilidad_principal):

            # Asigna la categoría de ausencia de comorbilidad.
            comorbilidad_principal = "SIN_COMORBILIDAD"


        # Devuelve las tres variables derivadas calculadas.
        return pd.Series([
            cancer,
            comorbilidad,
            comorbilidad_principal
        ])


    # Aplica el procesamiento de diagnósticos a cada fila del dataset.
    df[
        [
            'CARGA_ONCOLOGICA',
            'NUM_COMORBILIDADES',
            'COMORBILIDAD_PRINCIPAL'
        ]
    ] = df.apply(
        procesar_diagnosticos,
        axis=1
    )


    # =========================================================================
    # D. MANEJO DE FECHAS: EDAD Y DÍAS DE ESTADÍA
    # =========================================================================

    # Convierte la fecha de nacimiento al formato datetime.
    df['FECHA_NACIMIENTO_DT'] = pd.to_datetime(
        df['FECHA_NACIMIENTO'],
        format='%Y-%m-%d',
        errors='coerce'
    )


    # Determina el formato de fecha utilizado para cada año.
    if año == '2023':

        # Define el formato utilizado durante 2023.
        formato_operativo = '%d-%m-%Y'

    else:

        # Define el formato utilizado en los demás años.
        formato_operativo = '%Y-%m-%d'


    # Convierte la fecha de ingreso al formato datetime.
    df['FECHA_INGRESO_DT'] = pd.to_datetime(
        df['FECHA_INGRESO'],
        format=formato_operativo,
        errors='coerce'
    )

    # Convierte la fecha de alta al formato datetime.
    df['FECHAALTA_DT'] = pd.to_datetime(
        df['FECHAALTA'],
        format=formato_operativo,
        errors='coerce'
    )


    # =========================================================================
    # CÁLCULO DE EDAD Y DÍAS DE ESTADÍA
    # =========================================================================

    # Calcula la edad aproximada del paciente al momento del ingreso.
    df['EDAD'] = np.floor(
        (
            df['FECHA_INGRESO_DT']
            - df['FECHA_NACIMIENTO_DT']
        ).dt.days / 365.25
    )

    # Calcula la cantidad de días entre el ingreso y el alta.
    df['DIAS_ESTADIA'] = (
        df['FECHAALTA_DT']
        - df['FECHA_INGRESO_DT']
    ).dt.days


    # =========================================================================
    # LIMPIEZA FINAL DE ANOMALÍAS TEMPORALES
    # =========================================================================

    # Cuenta las estadías con valores negativos.
    n_estadia_neg = (
        df['DIAS_ESTADIA'] < 0
    ).sum()

    # Cuenta las edades con valores negativos.
    n_edad_neg = (
        df['EDAD'] < 0
    ).sum()


    # Informa si se encontraron anomalías temporales.
    if n_estadia_neg > 0 or n_edad_neg > 0:

        # Muestra la cantidad de anomalías detectadas.
        print(
            f"  - Limpiando anomalías reales en {año}: "
            f"{n_estadia_neg} estadías < 0, "
            f"{n_edad_neg} edades < 0."
        )


    # Convierte las estadías negativas en valores nulos.
    df.loc[
        df['DIAS_ESTADIA'] < 0,
        'DIAS_ESTADIA'
    ] = np.nan

    # Convierte las edades negativas en valores nulos.
    df.loc[
        df['EDAD'] < 0,
        'EDAD'
    ] = np.nan


    # =========================================================================
    # ELIMINACIÓN DE COLUMNAS TEMPORALES Y ORIGINALES
    # =========================================================================

    # Elimina las columnas temporales utilizadas para calcular las variables.
    df.drop(
        columns=[
            'FECHA_NACIMIENTO_DT',
            'FECHA_INGRESO_DT',
            'FECHAALTA_DT'
        ],
        inplace=True
    )

    # Elimina las columnas de fechas originales después de utilizarlas.
    df.drop(
        columns=[
            'FECHA_NACIMIENTO',
            'FECHA_INGRESO',
            'FECHAALTA'
        ],
        inplace=True
    )


    # =========================================================================
    # GUARDAR NUEVO DATASET
    # =========================================================================

    # Genera el nombre del nuevo archivo con las variables derivadas.
    nuevo_nombre = (
        f"GRD_PROCESADO_{año}_DERIVADAS.csv"
    )

    # Construye la ruta completa del archivo de destino.
    ruta_destino = os.path.join(
        carpeta_procesados,
        nuevo_nombre
    )

    # Guarda el dataset sin eliminar las demás columnas existentes.
    df.to_csv(
        ruta_destino,
        index=False,
        encoding="utf-8-sig"
    )

    # Informa que el archivo fue generado correctamente.
    print(
        f"[{año}] Guardado: {nuevo_nombre}"
    )


# ============================================================================
# FINALIZACIÓN DEL PROCESO
# ============================================================================

# Informa que terminó la generación de todas las variables derivadas.
print(
    "=" * 60
    + "\nProceso de generación de variables derivadas completado"
)

Iniciando creación de variables derivadas...
[2019] Guardado: GRD_PROCESADO_2019_DERIVADAS.csv
  - Limpiando anomalías reales en 2020: 10 estadías < 0, 0 edades < 0.
[2020] Guardado: GRD_PROCESADO_2020_DERIVADAS.csv
[2021] Guardado: GRD_PROCESADO_2021_DERIVADAS.csv
  - Limpiando anomalías reales en 2022: 1 estadías < 0, 0 edades < 0.
[2022] Guardado: GRD_PROCESADO_2022_DERIVADAS.csv
[2023] Guardado: GRD_PROCESADO_2023_DERIVADAS.csv
[2024] Guardado: GRD_PROCESADO_2024_DERIVADAS.csv
============================================================Proceso de generación de variables derivadas completado


### Procesamiento

In [ ]:
import pandas as pd
import numpy as np
import os
import re
from IPython.display import display, HTML

# 1. Definir rutas
carpeta_procesados = "../../Datos/Datos procesados"
archivos_derivadas = [f"GRD_PROCESADO_{año}_DERIVADAS.csv" for año in range(2019, 2025)]

# ELIMINAMOS ES_QUIRURGICO DE AQUÍ
vars_cat = ['COMORBILIDAD_PRINCIPAL']
vars_num = ['CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 'NUM_COMORBILIDADES', 
            'NUM_PROCEDIMIENTOS', 'EDAD', 'DIAS_ESTADIA']

resultados_cat = {var: [] for var in vars_cat}
resultados_num = []

print("Leyendo datasets y calculando estadísticas... \n")

for archivo in archivos_derivadas:
    ruta = os.path.join(carpeta_procesados, archivo)
    if not os.path.exists(ruta):
        continue
        
    año = re.search(r'\d{4}', archivo).group()
    df = pd.read_csv(ruta, low_memory=False)
    
    # --- 1. CATEGÓRICAS ---
    for var in vars_cat:
        if var in df.columns:
            # Calcular conteos absolutos y porcentajes
            conteos = df[var].value_counts(dropna=False)
            porcentajes = df[var].value_counts(dropna=False, normalize=True) * 100
            
            # Formatear el texto combinando ambos "N (P%)"
            freq = pd.DataFrame({
                var: conteos.index,
                'Valor_Formateado': conteos.astype(str) + " (" + porcentajes.round(2).astype(str) + "%)"
            })
            freq['Año'] = año
            resultados_cat[var].append(freq)
            
    # --- 2. NUMÉRICAS ---
    for var in vars_num:
        if var in df.columns:
            serie = df[var].dropna()
            n_total = len(serie)
            if n_total == 0: continue
                
            Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
            IQR = Q3 - Q1
            lim_inf, lim_sup = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
            
            outliers_mask = (serie < lim_inf) | (serie > lim_sup)
            n_outliers = outliers_mask.sum()
            
            resultados_num.append({
                'Año': año, 'Variable': var, 'N_Válidos': n_total,
                'Mín': serie.min(), 'Máx': serie.max(),
                'Media': round(serie.mean(), 2), 'Desv_Est': round(serie.std(), 2),
                'Outliers (N)': n_outliers, 'Outliers (%)': (n_outliers / n_total * 100).round(2)
            })

# Consolidar los resultados en DataFrames globales para consultarlos después
df_num_final = pd.DataFrame(resultados_num) if resultados_num else pd.DataFrame()
dicc_cat_final = {}

for var in vars_cat:
    if resultados_cat[var]:
        df_cat = pd.concat(resultados_cat[var], ignore_index=True)
        # Al pivotear, si un año no tiene un valor, se rellena con "0 (0.0%)"
        dicc_cat_final[var] = df_cat.pivot(index=var, columns='Año', values='Valor_Formateado').fillna("0 (0.0%)")

print("- ÉXITO: Cálculos terminados")

Leyendo datasets y calculando estadísticas... 

- ÉXITO: Cálculos terminados


In [4]:
print("--- COMORBILIDAD PRINCIPAL ---")
display(dicc_cat_final['COMORBILIDAD_PRINCIPAL'])

--- COMORBILIDAD PRINCIPAL ---


Año,2019,2020,2021,2022,2023,2024
COMORBILIDAD_PRINCIPAL,,,,,,
CODIGOS PROVISIONALES (COVID),49 (0.0%),1386 (0.18%),1748 (0.22%),1608 (0.17%),584 (0.06%),371 (0.03%)
ENDOCRINAS Y METABOLICAS,8390 (0.75%),5391 (0.7%),6470 (0.8%),7555 (0.82%),8911 (0.86%),9821 (0.91%)
INFECCIOSAS Y PARASITARIAS,2260 (0.2%),1754 (0.23%),1757 (0.22%),1971 (0.21%),2418 (0.23%),2673 (0.25%)
MATERNO-INFANTILES Y CONGENITAS,571 (0.05%),475 (0.06%),377 (0.05%),516 (0.06%),579 (0.06%),575 (0.05%)
NO_APLICA,1016813 (91.42%),705708 (91.67%),741678 (91.63%),849421 (91.64%),940799 (91.26%),978090 (90.87%)
OJO Y OIDO,1064 (0.1%),540 (0.07%),655 (0.08%),1033 (0.11%),1292 (0.13%),1323 (0.12%)
PIEL Y TEJIDO SUBCUTANEO,905 (0.08%),644 (0.08%),659 (0.08%),714 (0.08%),878 (0.09%),1092 (0.1%)
SANGRE E INMUNIDAD,5889 (0.53%),4448 (0.58%),4262 (0.53%),4705 (0.51%),5558 (0.54%),6326 (0.59%)
SINTOMAS Y HALLAZGOS,2864 (0.26%),2332 (0.3%),2543 (0.31%),2788 (0.3%),3261 (0.32%),3690 (0.34%)


In [5]:
display(df_num_final)

,Año,Variable,N_Válidos,Mín,Máx,Media,Desv_Est,Outliers (N),Outliers (%)
0,2019,CANTIDAD_TRASLADOS,1112304,0.0,9.0,0.22,0.62,163802,14.73
1,2019,CARGA_ONCOLOGICA,1112304,0.0,8.0,0.15,0.45,135143,12.15
2,2019,NUM_COMORBILIDADES,1112304,0.0,35.0,3.19,2.76,53931,4.85
3,2019,NUM_PROCEDIMIENTOS,1112304,1.0,30.0,6.82,5.24,42307,3.80
4,2019,EDAD,1112304,0.0,124.0,43.09,25.83,0,0.00
5,2019,DIAS_ESTADIA,1112304,0.0,2322.0,5.30,12.85,121103,10.89
6,2020,CANTIDAD_TRASLADOS,769815,0.0,9.0,0.32,0.75,162427,21.10
7,2020,CARGA_ONCOLOGICA,769815,0.0,9.0,0.15,0.45,90034,11.70
8,2020,NUM_COMORBILIDADES,769815,0.0,35.0,3.89,3.38,28228,3.67
9,2020,NUM_PROCEDIMIENTOS,769815,1.0,30.0,8.40,5.97,33966,4.41


In [ ]:
import pandas as pd # Librería para manipulación de datos
import numpy as np # Librería para operaciones numéricas
import os # Librería para manejo de rutas y archivos
import re # Librería para expresiones regulares
from IPython.display import display, HTML # Librería para mostrar tablas en Jupyter Notebook


# ============================================================================
# 1. DEFINICIÓN DE RUTAS Y ARCHIVOS
# ============================================================================

# Define la carpeta donde se encuentran los datasets procesados.
carpeta_procesados = "../../Datos/Datos procesados"

# Genera los nombres de los archivos derivados correspondientes a 2019-2024.
archivos_derivadas = [
    f"GRD_PROCESADO_{año}_DERIVADAS.csv"
    for año in range(2019, 2025)
]


# ============================================================================
# 2. DEFINICIÓN DE VARIABLES A ANALIZAR
# ============================================================================

# Define las variables categóricas que serán analizadas.
vars_cat = [
    'COMORBILIDAD_PRINCIPAL'
]

# Define las variables numéricas que serán analizadas.
vars_num = [
    'CANTIDAD_TRASLADOS',
    'CARGA_ONCOLOGICA',
    'NUM_COMORBILIDADES',
    'NUM_PROCEDIMIENTOS',
    'EDAD',
    'DIAS_ESTADIA'
]


# ============================================================================
# 3. ESTRUCTURAS PARA ALMACENAR RESULTADOS
# ============================================================================

# Crea un diccionario para almacenar los resultados de las variables categóricas.
resultados_cat = {
    var: []
    for var in vars_cat
}

# Crea una lista para almacenar los resultados de las variables numéricas.
resultados_num = []


# ============================================================================
# 4. LECTURA Y ANÁLISIS DE LOS DATASETS
# ============================================================================

# Informa que comienza el proceso de cálculo de estadísticas.
print(
    "Leyendo datasets y calculando estadísticas... \n"
)


# Recorre todos los archivos derivados definidos anteriormente.
for archivo in archivos_derivadas:

    # Construye la ruta completa del archivo.
    ruta = os.path.join(
        carpeta_procesados,
        archivo
    )

    # Verifica que el archivo exista antes de intentar leerlo.
    if not os.path.exists(ruta):

        # Si el archivo no existe, continúa con el siguiente.
        continue

    # Extrae el año desde el nombre del archivo mediante una expresión regular.
    año = re.search(
        r'\d{4}',
        archivo
    ).group()

    # Carga el dataset completo en un DataFrame.
    df = pd.read_csv(
        ruta,
        low_memory=False
    )


    # =========================================================================
    # 5. ANÁLISIS DE VARIABLES CATEGÓRICAS
    # =========================================================================

    # Recorre todas las variables categóricas definidas.
    for var in vars_cat:

        # Verifica que la variable exista en el dataset.
        if var in df.columns:

            # Calcula los conteos absolutos incluyendo los valores nulos.
            conteos = df[var].value_counts(
                dropna=False
            )

            # Calcula los porcentajes de cada categoría incluyendo los nulos.
            porcentajes = (
                df[var]
                .value_counts(
                    dropna=False,
                    normalize=True
                )
                * 100
            )

            # Combina cantidad y porcentaje en un formato "N (P%)".
            freq = pd.DataFrame({
                var: conteos.index,
                'Valor_Formateado': (
                    conteos.astype(str)
                    + " ("
                    + porcentajes.round(2).astype(str)
                    + "%)"
                )
            })

            # Agrega el año correspondiente a los resultados.
            freq['Año'] = año

            # Guarda los resultados de la variable categórica.
            resultados_cat[var].append(freq)


    # =========================================================================
    # 6. ANÁLISIS DE VARIABLES NUMÉRICAS
    # =========================================================================

    # Recorre todas las variables numéricas definidas.
    for var in vars_num:

        # Verifica que la variable exista en el dataset.
        if var in df.columns:

            # Elimina los valores nulos antes de calcular las estadísticas.
            serie = df[var].dropna()

            # Calcula la cantidad de observaciones válidas.
            n_total = len(serie)

            # Si no existen observaciones válidas, continúa con la siguiente variable.
            if n_total == 0:
                continue


            # =================================================================
            # CÁLCULO DEL RANGO INTERCUARTÍLICO
            # =================================================================

            # Calcula el primer cuartil de la variable.
            Q1 = serie.quantile(0.25)

            # Calcula el tercer cuartil de la variable.
            Q3 = serie.quantile(0.75)

            # Calcula el rango intercuartílico.
            IQR = Q3 - Q1

            # Calcula el límite inferior para identificar valores atípicos.
            lim_inf = Q1 - 1.5 * IQR

            # Calcula el límite superior para identificar valores atípicos.
            lim_sup = Q3 + 1.5 * IQR


            # =================================================================
            # IDENTIFICACIÓN DE OUTLIERS
            # =================================================================

            # Identifica valores inferiores o superiores a los límites definidos.
            outliers_mask = (
                (serie < lim_inf)
                | (serie > lim_sup)
            )

            # Cuenta la cantidad de valores considerados outliers.
            n_outliers = outliers_mask.sum()


            # =================================================================
            # ALMACENAMIENTO DE ESTADÍSTICAS
            # =================================================================

            # Guarda las estadísticas calculadas para la variable y año.
            resultados_num.append({
                'Año': año,
                'Variable': var,
                'N_Válidos': n_total,
                'Mín': serie.min(),
                'Máx': serie.max(),
                'Media': round(
                    serie.mean(),
                    2
                ),
                'Desv_Est': round(
                    serie.std(),
                    2
                ),
                'Outliers (N)': n_outliers,
                'Outliers (%)': (
                    n_outliers
                    / n_total
                    * 100
                ).round(2)
            })


# ============================================================================
# 7. CONSOLIDACIÓN DE RESULTADOS NUMÉRICOS
# ============================================================================

# Convierte los resultados numéricos en un DataFrame global.
df_num_final = (
    pd.DataFrame(resultados_num)
    if resultados_num
    else pd.DataFrame()
)


# ============================================================================
# 8. CONSOLIDACIÓN DE RESULTADOS CATEGÓRICOS
# ============================================================================

# Inicializa el diccionario que almacenará las tablas categóricas finales.
dicc_cat_final = {}


# Recorre todas las variables categóricas analizadas.
for var in vars_cat:

    # Verifica si existen resultados para la variable actual.
    if resultados_cat[var]:

        # Combina los resultados de todos los años en un único DataFrame.
        df_cat = pd.concat(
            resultados_cat[var],
            ignore_index=True
        )

        # Convierte los resultados a formato ancho con un año por columna.
        dicc_cat_final[var] = (
            df_cat
            .pivot(
                index=var,
                columns='Año',
                values='Valor_Formateado'
            )
            .fillna("0 (0.0%)")
        )


# ============================================================================
# 9. FINALIZACIÓN DEL PROCESO
# ============================================================================

# Informa que los cálculos estadísticos finalizaron correctamente.
print(
    "- ÉXITO: Cálculos terminados"
)

Leyendo y calculando nulos...

=== TOTAL DE EGRESOS POR AÑO ===
2019: 1.112.304
2020: 769.815
2021: 809.441
2022: 926.931
2023: 1.030.863
2024: 1.076.398

=== RECUENTO DE NULOS (NaN) EN VARIABLES DERIVADAS ===


Año,2019,2020,2021,2022,2023,2024
Variable,,,,,,
CANTIDAD_TRASLADOS,0,0,0,0,0,0
CARGA_ONCOLOGICA,0,0,0,0,0,0
DIAS_ESTADIA,0,10 (0.001%),0,1 (0.000%),0,50 (0.005%)
EDAD,0,1 (0.000%),0,7 (0.001%),10 (0.001%),53 (0.005%)
NUM_COMORBILIDADES,0,0,0,0,0,0
NUM_PROCEDIMIENTOS,0,0,0,0,0,0


In [7]:
import pandas as pd
import os
import re

# 1. Definir rutas
carpeta_procesados = "../../Datos/Datos procesados"
archivos_derivadas = [f"GRD_PROCESADO_{año}_DERIVADAS.csv" for año in range(2019, 2025)]

print("Iniciando eliminación de valores nulos residuales...\n" + "="*60)

# Variables donde no permitiremos ningún NaN
columnas_criticas = ['EDAD', 'DIAS_ESTADIA']

for archivo in archivos_derivadas:
    ruta = os.path.join(carpeta_procesados, archivo)
    if not os.path.exists(ruta):
        continue
        
    # Extracción segura del año usando Regex
    año = re.search(r'\d{4}', archivo).group()
    
    # Leer el dataset
    df = pd.read_csv(ruta, low_memory=False)
    filas_iniciales = len(df)
    
    # Eliminar filas que tengan NaN en las columnas críticas
    df.dropna(subset=columnas_criticas, inplace=True)
    
    filas_finales = len(df)
    eliminados = filas_iniciales - filas_finales
    
    # Guardar el dataset limpio sobreescribiendo el archivo
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    
    print(f"[{año}] Pacientes iniciales: {filas_iniciales:,}".replace(',', '.'))
    print(f"       Pacientes eliminados: {eliminados} filas con NaN")
    print(f"       Pacientes finales: {filas_finales:,} (Guardado)\n".replace(',', '.'))

print("="*60 + "\nÉXITO: Limpieza completada (dataset libre de NULOS).")

Iniciando eliminación de valores nulos residuales...
[2019] Pacientes iniciales: 1.112.304
       Pacientes eliminados: 0 filas con NaN
       Pacientes finales: 1.112.304 (Guardado)

[2020] Pacientes iniciales: 769.815
       Pacientes eliminados: 11 filas con NaN
       Pacientes finales: 769.804 (Guardado)

[2021] Pacientes iniciales: 809.441
       Pacientes eliminados: 0 filas con NaN
       Pacientes finales: 809.441 (Guardado)

[2022] Pacientes iniciales: 926.931
       Pacientes eliminados: 8 filas con NaN
       Pacientes finales: 926.923 (Guardado)

[2023] Pacientes iniciales: 1.030.863
       Pacientes eliminados: 10 filas con NaN
       Pacientes finales: 1.030.853 (Guardado)

[2024] Pacientes iniciales: 1.076.398
       Pacientes eliminados: 53 filas con NaN
       Pacientes finales: 1.076.345 (Guardado)

ÉXITO: Limpieza completada (dataset libre de NULOS).
